In [ ]:
!pip install -q pymysql sqlalchemy

import time
import json
from datetime import datetime, date, timezone
from pathlib import Path

import requests as req
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.0 MB/s eta 0:00:00


84be1a77a79e4d7f9ced1c607bf2ad7c

In [ ]:
from getpass import getpass

def obtener_secreto(nombre, prompt):
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor:
            return valor
    except Exception:
        pass
    return getpass(prompt)

NEWS_API_KEY = obtener_secreto("NEWS_API_KEY", "Pega tu NewsAPI key: ")
DB_HOST = obtener_secreto("DB_HOST", "IP de tu instancia de MariaDB: ")
DB_USER = obtener_secreto("DB_USER", "Usuario de MariaDB: ")
DB_PASSWORD = obtener_secreto("DB_PASSWORD", "Password de MariaDB: ")
DB_NAME = "prices"
DB_PORT = 3306

Pega tu NewsAPI key: ··········
IP de tu instancia de MariaDB: ··········
Usuario de MariaDB: ··········
Password de MariaDB: ··········


In [ ]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)


def extract_exchange_rate(moneda_destino="EUR"):
    """Tipo de cambio USD -> moneda_destino, vía Frankfurter (sin API key)."""
    url = "https://api.frankfurter.dev/v1/latest"
    resp = req.get(url, params={"base": "USD", "symbols": moneda_destino}, timeout=10)
    resp.raise_for_status()
    return resp.json()


def extract_market_snapshot(per_page=30):
    """Top `per_page` criptomonedas por market cap, vía CoinGecko (sin API key)."""
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": per_page,
        "price_change_percentage": "24h",
    }
    resp = req.get(url, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()


def guardar_crudo(datos, nombre):
    ruta = RAW_DIR / f"{nombre}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    return ruta

In [ ]:
extract_exchange_rate()

{'amount': 1.0, 'base': 'USD', 'date': '2026-10-19', 'rates': {'EUR': 0.89087}}

In [ ]:
extract_market_snapshot()

[{'id': 'bitcoin',
  'symbol': 'btc',
  'name': 'Bitcoin',
  'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
  'current_price': 81728,
  'market_cap': 1642548449813,
  'market_cap_rank': 1,
  'fully_diluted_valuation': 1642548449813,
  'total_volume': 25573281534,
  'high_24h': 83035,
  'low_24h': 81155,
  'price_change_24h': -901.7413353414158,
  'price_change_percentage_24h': -1.11631,
  'market_cap_change_24h': -18211987873.853516,
  'market_cap_change_percentage_24h': -1.09661,
  'circulating_supply': 20098843.0,
  'total_supply': 20098843.0,
  'max_supply': 21000000.0,
  'ath': 126080,
  'ath_change_percentage': -35.17768,
  'ath_date': '2025-10-06T10:57:42.000Z',
  'atl': 67.81,
  'atl_change_percentage': 120426.64455,
  'atl_date': '2013-07-05T16:00:00.000Z',
  'roi': None,
  'last_updated': '2026-10-20T02:52:52.000Z',
  'price_change_percentage_24h_in_currency': -1.11631},
 {'id': 'ethereum',
  'symbol': 'eth',
  'name': 'Ethereum',
  '

In [ ]:
raw_fx = extract_exchange_rate("EUR")
raw_market = extract_market_snapshot(per_page=30)

guardar_crudo(raw_fx, "tipo_cambio")
guardar_crudo(raw_market, "mercado_cripto")

eur_rate = raw_fx["rates"]["EUR"]
print(f"EXTRACT OK — tipo de cambio USD/EUR: {eur_rate} | {len(raw_market)} monedas extraídas")

EXTRACT OK — tipo de cambio USD/EUR: 0.89087 | 30 monedas extraídas


In [ ]:
eur_rate

0.89087

In [ ]:
def transform_market_data(raw_list, eur_rate, fecha_snapshot):
    """Limpia el JSON de CoinGecko y calcula columnas derivadas. No llama a ninguna API."""
    if not raw_list:
        return pd.DataFrame(), pd.DataFrame(), []

    df = pd.DataFrame(raw_list)

    cols_map = {
        "id": "coin_id", "symbol": "symbol", "name": "name",
        "current_price": "price_usd", "price_change_percentage_24h": "change_24h_pct",
        "total_volume": "volume_24h_usd", "market_cap": "market_cap_usd",
    }
    df_clean = df[list(cols_map.keys())].rename(columns=cols_map).copy()

    # --- Tabla de dimensión (catálogo de monedas, sin duplicar en cada snapshot) ---
    df_dim_moneda = df_clean[["coin_id", "symbol", "name"]].drop_duplicates()

    # --- Tabla de hechos (el snapshot de mercado de este momento) ---
    df_hechos = df_clean.drop(columns=["symbol", "name"]).copy()
    df_hechos["fecha_snapshot"] = fecha_snapshot
    df_hechos["price_eur"] = df_hechos["price_usd"] * eur_rate
    df_hechos["is_alert"] = df_hechos["change_24h_pct"].apply(lambda x: abs(x or 0) >= 5)

    # --- Decisión que alimenta el segundo Extract: quiénes son "top movers" ---
    top_movers_ids = pd.concat([
        df_clean.nlargest(5, "change_24h_pct"),
        df_clean.nsmallest(5, "change_24h_pct"),
    ])["coin_id"].unique().tolist()

    return df_dim_moneda, df_hechos, top_movers_ids

In [ ]:
fecha_snapshot = datetime.now(timezone.utc).replace(second=0, microsecond=0, tzinfo=None)

df_dim_moneda, df_hechos_mercado, top_movers_ids = transform_market_data(
    raw_market, eur_rate, fecha_snapshot
)

print(f"TRANSFORM OK — {len(df_hechos_mercado)} monedas limpias, "
      f"{len(top_movers_ids)} identificadas como top movers")
df_hechos_mercado

TRANSFORM OK — 30 monedas limpias, 10 identificadas como top movers


,coin_id,price_usd,change_24h_pct,volume_24h_usd,market_cap_usd,fecha_snapshot,price_eur,is_alert
0,bitcoin,81737.000000,-1.07989,25539704544,1642766909745,2026-10-20 03:05:00,72817.041190,False
1,ethereum,2398.860000,-1.14620,9194582253,293001577681,2026-10-20 03:05:00,2137.072408,False
2,tether,0.999169,-0.00006,46891846130,183024072437,2026-10-20 03:05:00,0.890130,False
3,binancecoin,728.740000,0.19579,717092679,95838021338,2026-10-20 03:05:00,649.212604,False
4,ripple,1.300000,-0.79129,1302021559,82232642620,2026-10-20 03:05:00,1.158131,False
5,usd-coin,0.999920,0.01857,16455892048,72120449742,2026-10-20 03:05:00,0.890799,False
6,solana,105.030000,-0.42282,2583221811,61910410936,2026-10-20 03:05:00,93.568076,False
7,tron,0.331599,-0.25936,326133141,31502174224,2026-10-20 03:05:00,0.295412,False
8,figure-heloc,1.021000,NaN,18947467,24529246843,2026-10-20 03:05:00,0.909578,False
9,zcash,1145.440000,-2.57947,686077385,19464410023,2026-10-20 03:05:00,1020.438133,False


In [ ]:
def extract_news_for_coin(coin_name):
    """Un artículo reciente para `coin_name`. Devuelve (titulo, url) o (None, None)."""
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": f"{coin_name} crypto",
        "sortBy": "publishedAt",
        "pageSize": 1,
        "apiKey": NEWS_API_KEY,
    }
    try:
        resp = req.get(url, params=params, timeout=5)
        data = resp.json()
        if data.get("status") == "ok" and data.get("articles"):
            art = data["articles"][0]
            return art["title"], art["url"]
    except req.exceptions.RequestException as e:
        print(f"  aviso: no se pudo consultar noticias de {coin_name}: {e}")
    return None, None

In [ ]:
nombres_top_movers = df_dim_moneda[
    df_dim_moneda["coin_id"].isin(top_movers_ids)
][["coin_id", "name"]]

raw_noticias = []
for _, fila in nombres_top_movers.iterrows():
    titulo, link = extract_news_for_coin(fila["name"])
    raw_noticias.append({"coin_id": fila["coin_id"], "news_title": titulo, "news_url": link})
    time.sleep(0.2)  # cuidar la cuota gratuita de NewsAPI

guardar_crudo(raw_noticias, "noticias_cripto")
print(f"EXTRACT OK — {len(raw_noticias)} monedas consultadas en NewsAPI")

EXTRACT OK — 10 monedas consultadas en NewsAPI


In [ ]:
nombres_top_movers

,coin_id,name
3,binancecoin,BNB
9,zcash,Zcash
12,monero,Monero
13,usds,USDS
17,cardano,Cardano
18,near,NEAR Protocol
20,rain,Rain
21,bitcoin-cash,Bitcoin Cash
26,canton-network,Canton
27,avalanche-2,Avalanche


In [ ]:
raw_noticias

[{'coin_id': 'binancecoin',
  'news_title': 'Why Binance’s Changpeng Zhao Is Throwing Support Behind XRP, Solana, Zcash Even As China Rattles Markets',
  'news_url': 'https://zycrypto.com/why-binances-changpeng-zhao-is-throwing-support-behind-xrp-solana-zcash-even-as-china-rattles-markets/'},
 {'coin_id': 'zcash',
  'news_title': 'ZEC cae 5,4% y pone a prueba soportes clave el 18 de octubre de 2026',
  'news_url': 'https://www.diariobitcoin.com/analisis/zec-cae-54-y-pone-a-prueba-soportes-clave-el-18-de-octubre-de-2026/'},
 {'coin_id': 'monero',
  'news_title': 'Richard Heart caught with pants down, QDay gets even closer: Hodler’s Digest',
  'news_url': 'https://cointelegraph.com/magazine/richard-heart-caught-with-pants-down-qday-gets-even-closer-hodlers-digest'},
 {'coin_id': 'usds',
  'news_title': 'USDC Dominates AI Agent Payments As Circulating Supply Stays Above MiCA’s €5B Threshold For 12 Months',
  'news_url': 'https://zycrypto.com/usdc-dominates-ai-agent-payments-as-circulating

In [ ]:
def transform_noticias(raw_noticias, fecha_snapshot):
    df = pd.DataFrame(raw_noticias)
    df = df.dropna(subset=["news_title", "news_url"])  # descartar donde no hubo artículo real
    df["fecha_snapshot"] = fecha_snapshot
    return df[["coin_id", "fecha_snapshot", "news_title", "news_url"]]


def transform_tipo_cambio(raw_fx):
    fecha = pd.to_datetime(raw_fx["date"]).date()
    filas = [
        {"fecha": fecha, "moneda_origen": raw_fx["base"], "moneda_destino": moneda, "tasa": tasa}
        for moneda, tasa in raw_fx["rates"].items()
    ]
    return pd.DataFrame(filas)


df_noticias = transform_noticias(raw_noticias, fecha_snapshot)
df_tipo_cambio = transform_tipo_cambio(raw_fx)

print(f"TRANSFORM OK — {len(df_noticias)} noticias reales (de {len(raw_noticias)} monedas consultadas)")
df_noticias

TRANSFORM OK — 10 noticias reales (de 10 monedas consultadas)


,coin_id,fecha_snapshot,news_title,news_url
0,binancecoin,2026-10-20 03:05:00,Why Binance’s Changpeng Zhao Is Throwing Suppo...,https://zycrypto.com/why-binances-changpeng-zh...
1,zcash,2026-10-20 03:05:00,"ZEC cae 5,4% y pone a prueba soportes clave el...",https://www.diariobitcoin.com/analisis/zec-cae...
2,monero,2026-10-20 03:05:00,"Richard Heart caught with pants down, QDay get...",https://cointelegraph.com/magazine/richard-hea...
3,usds,2026-10-20 03:05:00,USDC Dominates AI Agent Payments As Circulatin...,https://zycrypto.com/usdc-dominates-ai-agent-p...
4,cardano,2026-10-20 03:05:00,"Cardano Fell 12.6% This Week, More Than 6 Time...",https://biztoc.com/x/8f6ef10061e0ed61
5,near,2026-10-20 03:05:00,Stacks outlines Q4 priorities to grow BTC bond...,https://cryptobriefing.com/stacks-q4-btc-bondi...
6,rain,2026-10-20 03:05:00,Made in India isn’t enough. It has to be made ...,https://timesofindia.indiatimes.com/science/ma...
7,bitcoin-cash,2026-10-20 03:05:00,Upbit weekly trading volume falls 15.5% to 8.3...,https://cryptobriefing.com/upbit-weekly-tradin...
8,canton-network,2026-10-20 03:05:00,USDC Dominates AI Agent Payments As Circulatin...,https://zycrypto.com/usdc-dominates-ai-agent-p...
9,avalanche-2,2026-10-20 03:05:00,"Avalanche (AVAX) cae 4,66% y pone a prueba sop...",https://www.diariobitcoin.com/analisis/avalanc...


In [ ]:
def validar_calidad(df_hechos, df_tipo_cambio):
    errores = []

    if df_hechos["price_usd"].isnull().any():
        errores.append("Hay precios nulos en hechos_mercado.")
    if (df_hechos["price_usd"] < 0).any():
        errores.append("Hay precios negativos en hechos_mercado.")
    if df_hechos.duplicated(subset=["coin_id", "fecha_snapshot"]).any():
        errores.append("Hay filas duplicadas en hechos_mercado por (coin_id, fecha_snapshot).")
    if (df_tipo_cambio["tasa"] <= 0).any():
        errores.append("Hay una tasa de cambio <= 0.")

    if errores:
        for e in errores:
            print(f"  - {e}")
        raise ValueError("Los datos no pasaron la validación de calidad.")
    print("Validación de calidad: OK")


validar_calidad(df_hechos_mercado, df_tipo_cambio)

Validación de calidad: OK


In [ ]:
DB_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
ENGINE = create_engine(DB_URL)

SQL_CREATE_TABLAS = """
CREATE TABLE IF NOT EXISTS dim_moneda_cripto (
    coin_id VARCHAR(50) NOT NULL,
    symbol  VARCHAR(20) NOT NULL,
    name    VARCHAR(100) NOT NULL,
    PRIMARY KEY (coin_id)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS hechos_mercado_cripto (
    coin_id        VARCHAR(50)   NOT NULL,
    fecha_snapshot DATETIME      NOT NULL,
    price_usd      DECIMAL(18,6),
    price_eur      DECIMAL(18,6),
    volume_24h_usd DECIMAL(24,2),
    market_cap_usd DECIMAL(24,2),
    change_24h_pct DECIMAL(8,4),
    is_alert       TINYINT(1),
    PRIMARY KEY (coin_id, fecha_snapshot)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS noticias_cripto (
    coin_id        VARCHAR(50)  NOT NULL,
    fecha_snapshot DATETIME     NOT NULL,
    news_title     VARCHAR(500),
    news_url       VARCHAR(500),
    PRIMARY KEY (coin_id, fecha_snapshot)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS tipo_cambio (
    fecha             DATE           NOT NULL,
    moneda_origen     VARCHAR(3)     NOT NULL,
    moneda_destino    VARCHAR(3)     NOT NULL,
    tasa              DECIMAL(14,6)  NOT NULL,
    variacion_diaria  DECIMAL(14,6)  NULL,
    variacion_pct     DECIMAL(8,4)   NULL,
    promedio_movil_7d DECIMAL(14,6)  NULL,
    PRIMARY KEY (fecha, moneda_origen, moneda_destino)
) ENGINE=InnoDB;
"""

with ENGINE.begin() as conn:
    for statement in SQL_CREATE_TABLAS.strip().split(";"):
        if statement.strip():
            conn.exec_driver_sql(statement)

print("Tablas listas: dim_moneda_cripto, hechos_mercado_cripto, noticias_cripto, tipo_cambio")

Tablas listas: dim_moneda_cripto, hechos_mercado_cripto, noticias_cripto, tipo_cambio


In [ ]:
def upsert_dataframe(df, table_name, engine, update_cols=None):
    """
    Inserta `df` en `table_name` con ON DUPLICATE KEY UPDATE. La tabla debe existir
    con su llave primaria ya definida (ver SQL_CREATE_TABLAS arriba).
    """
    if df.empty:
        print(f"[{table_name}] nada que cargar (DataFrame vacío)")
        return 0

    metadata = MetaData()
    tabla = Table(table_name, metadata, autoload_with=engine)
    pk_cols = {c.name for c in tabla.primary_key}

    registros = df.to_dict(orient="records")
    stmt = mysql_insert(tabla).values(registros)

    columnas_actualizar = update_cols or [c.name for c in tabla.columns if c.name not in pk_cols]
    stmt = stmt.on_duplicate_key_update({col: stmt.inserted[col] for col in columnas_actualizar})

    with engine.begin() as conn:
        conn.execute(stmt)

    print(f"[{table_name}] {len(registros)} filas cargadas (upsert)")
    return len(registros)

In [ ]:
df_hechos_mercado

,coin_id,price_usd,change_24h_pct,volume_24h_usd,market_cap_usd,fecha_snapshot,price_eur,is_alert
0,bitcoin,81737.000000,-1.07989,25539704544,1642766909745,2026-10-20 03:05:00,72817.041190,False
1,ethereum,2398.860000,-1.14620,9194582253,293001577681,2026-10-20 03:05:00,2137.072408,False
2,tether,0.999169,-0.00006,46891846130,183024072437,2026-10-20 03:05:00,0.890130,False
3,binancecoin,728.740000,0.19579,717092679,95838021338,2026-10-20 03:05:00,649.212604,False
4,ripple,1.300000,-0.79129,1302021559,82232642620,2026-10-20 03:05:00,1.158131,False
5,usd-coin,0.999920,0.01857,16455892048,72120449742,2026-10-20 03:05:00,0.890799,False
6,solana,105.030000,-0.42282,2583221811,61910410936,2026-10-20 03:05:00,93.568076,False
7,tron,0.331599,-0.25936,326133141,31502174224,2026-10-20 03:05:00,0.295412,False
8,figure-heloc,1.021000,NaN,18947467,24529246843,2026-10-20 03:05:00,0.909578,False
9,zcash,1145.440000,-2.57947,686077385,19464410023,2026-10-20 03:05:00,1020.438133,False


In [ ]:
upsert_dataframe(df_dim_moneda, "dim_moneda_cripto", ENGINE)
upsert_dataframe(df_hechos_mercado, "hechos_mercado_cripto", ENGINE)


[dim_moneda_cripto] 30 filas cargadas (upsert)


ProgrammingError: (pymysql.err.ProgrammingError) nan can not be used with MySQL
[SQL: INSERT INTO hechos_mercado_cripto (coin_id, fecha_snapshot, price_usd, price_eur, volume_24h_usd, market_cap_usd, change_24h_pct, is_alert) VALUES (%(coin_id_m0)s, %(fecha_snapshot_m0)s, %(price_usd_m0)s, %(price_eur_m0)s, %(volume_24h_usd_m0)s, %(market_cap_usd_m0)s, %(change_24h_pct_m0)s, %(is_alert_m0)s), (%(coin_id_m1)s, %(fecha_snapshot_m1)s, %(price_usd_m1)s, %(price_eur_m1)s, %(volume_24h_usd_m1)s, %(market_cap_usd_m1)s, %(change_24h_pct_m1)s, %(is_alert_m1)s), (%(coin_id_m2)s, %(fecha_snapshot_m2)s, %(price_usd_m2)s, %(price_eur_m2)s, %(volume_24h_usd_m2)s, %(market_cap_usd_m2)s, %(change_24h_pct_m2)s, %(is_alert_m2)s), (%(coin_id_m3)s, %(fecha_snapshot_m3)s, %(price_usd_m3)s, %(price_eur_m3)s, %(volume_24h_usd_m3)s, %(market_cap_usd_m3)s, %(change_24h_pct_m3)s, %(is_alert_m3)s), (%(coin_id_m4)s, %(fecha_snapshot_m4)s, %(price_usd_m4)s, %(price_eur_m4)s, %(volume_24h_usd_m4)s, %(market_cap_usd

In [ ]:
upsert_dataframe(df_noticias, "noticias_cripto", ENGINE)
upsert_dataframe(df_tipo_cambio, "tipo_cambio", ENGINE)

[noticias_cripto] 10 filas cargadas (upsert)
[tipo_cambio] 1 filas cargadas (upsert)


1